In [1]:
##### Analysis of wind forecasts for summer 2025 #####

#      - Comparison of GR and NWP with measured values (FOA)
#      - GR has not yet trained on measured values 
#      - Created by David Skrovanek, July 2025 

import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.dates import DateFormatter
import numpy as np
import os 
%matplotlib qt

plt.style.use('seaborn-v0_8-whitegrid')  
plt.rcParams['font.family'] = 'Arial' 

output_folder = "foa_dump"
os.makedirs(output_folder, exist_ok=True)

In [2]:
### Read raw input data files ###

# Read the GR Parquet file (in UTC)
df_GR = pd.read_parquet('./datasets/Summer 2025 analysis/From GR/FORM_inference_2025july.pq')
locations = pd.read_parquet('./datasets/Summer 2025 analysis/From GR/FORM_new_station_locations.pq')
locations['short_name'] = ['GRS-BAE M7', 'GRS-BAE M14', 'GRS-BAE M31', 'GRS-BAE M54', 'HHN-HHO M14', 'HHN-HHO M28', 'HHN-HHO M44', 'HHN-HHO M59', 'HHN-HHO M80', 'HHN-HHO M91', 'SM-DDS M9', 'SM-DDS M29', 'SM-DDS M54', 'SM-DDS M78']
df_GR['Datetime'] = pd.to_datetime(df_GR['time'])

# Read the FOA .csv file 
df_FOA_GRS_BAE = pd.read_csv('./datasets/Summer 2025 analysis/FOAs/GRS-BAE.csv')
df_FOA_DD_SM = pd.read_csv('./datasets/Summer 2025 analysis/FOAs/DD-SM.csv')
df_FOA_HHN_HHO = pd.read_csv('./datasets/Summer 2025 analysis/FOAs/HHN-HHO.csv')

In [3]:
### Take hourly average of FOA data ###

hourly_avg_wind_speed = {}
foa_dfs = [
    ('df_FOA_GRS_BAE', df_FOA_GRS_BAE, ['wind_m7', 'wind_m14', 'wind_m31', 'wind_m54'], 'hourly_avg_GRS_BAE'),
    ('df_FOA_DD_SM', df_FOA_DD_SM, ['Wind M9', 'Wind M29', 'Wind M54', 'Wind M78'], 'hourly_avg_DD_SM'),
    ('df_FOA_HHN_HHO', df_FOA_HHN_HHO, ['Wind M14', 'Wind M28', 'Wind M44', 'Wind M59', 'Wind M80', 'Wind M91'], 'hourly_avg_HHN_HHO'),
]

for name, df, wind_cols, out_name in foa_dfs:

    # Set timestamp as index
    df['Time'] = pd.to_datetime(df['Time'])
    df.set_index('Time', inplace=True)
    df.index = df.index + pd.DateOffset(hours=-7)  # time needs to be shifted by -7 hours

    # Clean text from data columns (each entry has 'm/s')
    for col in wind_cols:
        df[col] = (
            df[col]
            .astype(str)
            .str.replace(r'\s*m/s', '', regex=True)
            .astype(float)
        )
    # Resample all wind columns at once
    hourly_avg = df[wind_cols].resample('h', label='left', closed='left').mean().reset_index()
    hourly_avg_wind_speed[out_name] = hourly_avg

# Assign to standalone variables
hourly_avg_GRS_BAE = hourly_avg_wind_speed['hourly_avg_GRS_BAE']
hourly_avg_DD_SM = hourly_avg_wind_speed['hourly_avg_DD_SM']
hourly_avg_HHN_HHO = hourly_avg_wind_speed['hourly_avg_HHN_HHO']

# Save hourly_avg_GRS_BAE to .csv 
hourly_avg_GRS_BAE.to_csv(os.path.join(output_folder, "hourly_avg_GRS_BAE.csv"), index=False)
hourly_avg_HHN_HHO.to_csv(os.path.join(output_folder, "hourly_avg_HHN_HHO.csv"), index=False)
hourly_avg_DD_SM.to_csv(os.path.join(output_folder, "hourly_avg_DD_SM.csv"), index=False)

In [14]:
### Visual sanity check for GRS-BAE (making sure data aligns, no timestamp errors) ###

# Filter for desired time range to view 
start = pd.to_datetime('2025-05-21')
end = pd.to_datetime('2025-05-28')
df_pred_june = df_GR[(df_GR['Datetime'] >= start) & (df_GR['Datetime'] <= end)]

site_id = 111115  
df_pred_M7 = df_GR[df_GR['site_id'] == site_id].copy()
df_pred_M7['Datetime'] = pd.to_datetime(df_pred_M7['time'])
df_pred_M7_june = df_pred_M7[(df_pred_M7['Datetime'] >= start) & (df_pred_M7['Datetime'] <= end)]
df_pred_M7_june = df_pred_M7_june.sort_values('Datetime')

df_FOA_june = hourly_avg_GRS_BAE[
    (hourly_avg_GRS_BAE['Time'] >= start) & (hourly_avg_GRS_BAE['Time'] <= end)
]
df_FOA_M7_june = df_FOA_june[['Time', 'wind_m7']]

# Plot wind speed predictions vs NWP 
plt.figure(figsize=(12, 6))
plt.plot(df_pred_M7_june['Datetime'], df_pred_M7_june['wspeed_pred'], label='GR', linewidth=2)
plt.plot(df_pred_M7_june['Datetime'], df_pred_M7_june['wspeed_nwp'], label='NWP', linewidth=2)
plt.plot(df_FOA_M7_june['Time'], df_FOA_M7_june['wind_m7'], label='FOA (measured)', linewidth=2)
plt.ylim((0, 6))
plt.xlabel('Timestamp', fontsize=24, weight='bold')
plt.ylabel('Wind Speed (m/s)', fontsize=24, weight='bold')
plt.title(f'Wind Speed: Prediction vs NWP for site ID {site_id}\n{start.date()} to {end.date()}', fontsize=24)
plt.gca().tick_params(axis='both', which='both', labelsize=16)
plt.legend(frameon=True, facecolor='white', edgecolor='black', fontsize=16)
plt.grid(True, alpha=0.5)
ax = plt.gca()
plt.xlim([start, end])
ax.xaxis.set_major_formatter(DateFormatter('%m-%d'))
plt.show()

In [ ]:
### Mapping of site_id to column names in FOA datasets ###
col_to_siteid_GRS_BAE = {
    'wind_m7': 111115,
    'wind_m14': 111116,
    'wind_m31': 111117,
    'wind_m54': 111118,
}
col_to_siteid_HHN_HHO = {
    'Wind M14': 111119,
    'Wind M28': 111120,
    'Wind M44': 111121,
    'Wind M59': 111122,
    'Wind M80': 111123,
    'Wind M91': 111124
}
col_to_siteid_DD_SM = {
    'Wind M9': 111125,
    'Wind M29': 111126,
    'Wind M54': 111127,
    'Wind M78': 111128
}

foa_configs = [
    # (name, col_to_siteid_dict, hourly_avg_df)
    ("GRS-BAE", col_to_siteid_GRS_BAE, hourly_avg_GRS_BAE),
    ("DD-SM", col_to_siteid_DD_SM, hourly_avg_DD_SM),
    ("HHN-HHO", col_to_siteid_HHN_HHO, hourly_avg_HHN_HHO),
]

In [ ]:
### Make a histogram for wind speed differences (GR and FOAs) ###

# Set buckets for histogram (difference of wind speeds b/w -4 and 4 m/s with spacing of 1 m/s)
bin_edges = np.arange(-4, 5, 1)

for foa_name, col_to_siteid, hourly_avg_df in foa_configs:
    for foa_col, site_id in col_to_siteid.items():
        # Extract relevant data based on site
        df_gr_site = df_GR[df_GR['site_id'] == site_id][['Datetime', 'wspeed_pred']]
        df_foa_site = hourly_avg_df[['Time', foa_col]].rename(columns={foa_col: 'foa_wind'})
        
        # Merge data based on timestamp
        merged = pd.merge(
            df_gr_site, df_foa_site,
            left_on='Datetime', right_on='Time',
            how='inner',
            indicator=True
        )
        
        # Check for perfect merge of timestamps
        n_gr = len(df_gr_site)
        n_foa = len(df_foa_site)
        n_merged = len(merged)
        if n_merged < min(n_gr, n_foa):
            print(f"WARNING: Imperfect merge for site_id {site_id} ({foa_col}) in {foa_name}")
            print(f"  Rows in GR dataset: {n_gr}, Rows in FOA dataset: {n_foa}, Rows successfully merged: {n_merged}\n")
            unmatched_gr = set(df_gr_site['Datetime']) - set(merged['Datetime'])
            unmatched_foa = set(df_foa_site['Time']) - set(merged['Time'])
        else:
            print(f"Perfect merge for site_id {site_id} ({foa_col}) in {foa_name}")
        
        # Compute the difference (FOA - GR)
        diff = merged['foa_wind'] - merged['wspeed_pred']
        counts, bins = np.histogram(diff, bins=bin_edges)
        total_count = len(diff)
        percentages = (counts/total_count)*100

        plt.figure(figsize=(8,6))
        plt.bar(bin_edges[:-1], percentages, width=np.diff(bin_edges)[0], align='edge', alpha=1, edgecolor='white', color=(0, 0.4470, 0.7410))
        plt.title(f"{foa_name} | Site ID: {site_id} ({foa_col})", fontsize=24, weight='bold')
        plt.xlabel("FOA - GR (m/s)", fontsize=18, weight='bold')
        plt.ylabel("Occurence (%)", fontsize=18, weight='bold')
        plt.ylim((0, 70))
        plt.gca().yaxis.grid(True, alpha=0.7)   
        plt.gca().xaxis.grid(False)    
        plt.gca().tick_params(axis='both', which='major', labelsize=14)   


        # Compute and display descriptive statistics
        mean_val = np.mean(diff)
        median_val = np.median(diff)
        std_val = np.std(diff)
        stats_text = (
            f"Mean discrepancy: {mean_val:.2f} m/s\n"
            f"Median discrepancy: {median_val:.2f} m/s\n"
            f"σ of discrepancies: {std_val:.2f} m/s"
        )
        plt.gca().text(
            0.96, 0.96, stats_text,
            transform=plt.gca().transAxes,
            ha='right', va='top',
            bbox=dict(facecolor='white', edgecolor='white', boxstyle='round,pad=0.3'),
            fontsize=14)

        filename = f"GR_{foa_name}_site{site_id}_{foa_col}.png"
        filepath = os.path.join(output_folder, filename)
        plt.savefig(filepath, bbox_inches='tight', dpi=150)
        plt.show()

In [ ]:
### Make a histogram for wind speed differences (NWP and FOAs) ###

# Set buckets for histogram (difference of wind speeds b/w -4 and 4 m/s with spacing of 1 m/s)
bin_edges = np.arange(-4, 5, 1)

for foa_name, col_to_siteid, hourly_avg_df in foa_configs:
    for foa_col, site_id in col_to_siteid.items():
        # Extract relevant data based on site
        df_nwp_site = df_GR[df_GR['site_id'] == site_id][['Datetime', 'wspeed_nwp']]
        df_foa_site = hourly_avg_df[['Time', foa_col]].rename(columns={foa_col: 'foa_wind'})
        
        # Merge data based on timestamp
        merged = pd.merge(
            df_nwp_site, df_foa_site,
            left_on='Datetime', right_on='Time',
            how='inner',
            indicator=True
        )
        
        # Check for perfect merge of timestamp
        n_nwp = len(df_nwp_site)
        n_foa = len(df_foa_site)
        n_merged = len(merged)
        if n_merged < min(n_nwp, n_foa):
            print(f"WARNING: Imperfect merge for site_id {site_id} ({foa_col}) in {foa_name}")
            print(f"  Rows in NWP dataset: {n_nwp}, Rows in FOA dataset: {n_foa}, Rows successfully merged: {n_merged}\n")
            unmatched_nwp = set(df_nwp_site['Datetime']) - set(merged['Datetime'])
            unmatched_foa = set(df_foa_site['Time']) - set(merged['Time'])
        else:
            print(f"Perfect merge for site_id {site_id} ({foa_col}) in {foa_name}")
        
        # Compute the difference (FOA - GR)
        diff = merged['foa_wind'] - merged['wspeed_nwp']
        counts, bins = np.histogram(diff, bins=bin_edges)
        total_count = len(diff)
        percentages = (counts/total_count)*100

        plt.figure(figsize=(8,6))
        plt.bar(bin_edges[:-1], percentages, width=np.diff(bin_edges)[0], align='edge', alpha=1, edgecolor='white', color=(0.8500, 0.3250, 0.0980))
        plt.title(f"{foa_name} | Site ID: {site_id} ({foa_col})", fontsize=24, weight='bold')
        plt.xlabel("FOA - NWP (m/s)", fontsize=18, weight='bold')
        plt.ylabel("Occurence (%)", fontsize=18, weight='bold')
        plt.ylim((0, 70))
        plt.gca().yaxis.grid(True, alpha=0.7)   
        plt.gca().xaxis.grid(False)    
        plt.gca().tick_params(axis='both', which='major', labelsize=14)   


        # Compute and display descriptive statistics
        mean_val = np.mean(diff)
        median_val = np.median(diff)
        std_val = np.std(diff)
        stats_text = (
            f"Mean discrepancy: {mean_val:.2f} m/s\n"
            f"Median discrepancy: {median_val:.2f} m/s\n"
            f"σ of discrepancies: {std_val:.2f} m/s"
        )
        plt.gca().text(
            0.96, 0.96, stats_text,
            transform=plt.gca().transAxes,
            ha='right', va='top',
            fontsize=14)
        
        filename = f"NWP_{foa_name}_site{site_id}_{foa_col}.png"
        filepath = os.path.join(output_folder, filename)
        plt.savefig(filepath, bbox_inches='tight', dpi=150)
        plt.show()

In [ ]:
### Check how often measured values fall within 95% CI of GR predictions ###

# Define mappings for all locations
foa_configs = [
    # (location name, {foa_col: site_id, ...}, hourly_avg_df)
    ("GRS-BAE", {
        'wind_m7': 111115,
        'wind_m14': 111116,
        'wind_m31': 111117,
        'wind_m54': 111118,
    }, hourly_avg_GRS_BAE),
    ("DD-SM", {
        'Wind M9': 111125,
        'Wind M29': 111126,
        'Wind M54': 111127,
        'Wind M78': 111128,
    }, hourly_avg_DD_SM),
    ("HHN-HHO", {
        'Wind M14': 111119,
        'Wind M28': 111120,
        'Wind M44': 111121,
        'Wind M59': 111122,
        'Wind M80': 111123,
        'Wind M91': 111124,
    }, hourly_avg_HHN_HHO),
]

for loc_name, col_to_siteid, hourly_avg_df in foa_configs:
    print(f"\n--- {loc_name} ---")
    for foa_col, site_id in col_to_siteid.items():
        # Filter GR data for site
        df_gr_site = df_GR[df_GR['site_id'] == site_id][['Datetime', 'wspeed_lb_95', 'wspeed_ub_95']].copy()
        # FOA data for site
        df_foa_site = hourly_avg_df[['Time', foa_col]].rename(columns={'Time': 'Datetime', foa_col: 'foa_wind'})
        # Merge on timestamp
        merged = pd.merge(df_gr_site, df_foa_site, on='Datetime', how='inner')
        # Check if FOA value is within the 95% CI
        merged['within_95'] = (
            (merged['foa_wind'] >= merged['wspeed_lb_95']) &
            (merged['foa_wind'] <= merged['wspeed_ub_95'])
        )
        percent_within_95 = merged['within_95'].mean() * 100
        print(f"Site {site_id} ({foa_col}): FOA is within the 95% prediction interval {percent_within_95:.1f}% of the time.")

In [ ]:
### Visualizing measured values vs. confidence interval regions for GRS-BAE ###

# Set site and date range
site_id = 111115
start = pd.to_datetime('2025-05-21')
end = pd.to_datetime('2025-05-28')

# Filter GR data for site and date range
df_gr_plot = df_GR[(df_GR['site_id'] == site_id) & (df_GR['Datetime'] >= start) & (df_GR['Datetime'] <= end)].copy()

# Filter FOA data for date range
df_foa_plot = hourly_avg_GRS_BAE[(hourly_avg_GRS_BAE['Time'] >= start) & (hourly_avg_GRS_BAE['Time'] <= end)][['Time', 'wind_m7']].copy()

# Merge for aligned plotting
df_plot = pd.merge(
    df_gr_plot[['Datetime', 'wspeed_lb_95', 'wspeed_ub_95']],
    df_foa_plot.rename(columns={'Time': 'Datetime'}),
    on='Datetime',
    how='inner'
)

plt.figure(figsize=(12, 6))

# Shaded region for 95% interval
plt.fill_between(
    df_plot['Datetime'],
    df_plot['wspeed_lb_95'],
    df_plot['wspeed_ub_95'],
    color='gray',
    alpha=0.3,
    label='GR 95% Prediction Interval'
)

plt.plot(df_plot['Datetime'], df_plot['wind_m7'], label='FOA (measured)', color='tab:blue', linewidth=2)
plt.ylim((0, 6))
plt.xlim([start, end])
plt.xlabel('Timestamp', fontsize=24, weight='bold')
plt.ylabel('Wind Speed (m/s)', fontsize=24, weight='bold')
plt.title(f'FOA vs 95% Prediction Interval for site ID {site_id}\n{start.date()} to {end.date()}', fontsize=24)
plt.gca().tick_params(axis='both', which='both', labelsize=16)
plt.legend(frameon=True, facecolor='white', edgecolor='black', fontsize=16)
plt.grid(True, alpha=0.5)
ax = plt.gca()
ax.xaxis.set_major_formatter(DateFormatter('%m-%d'))
plt.show()